In [ ]:
from IPython.display import Image as IPImage
from IPython.display import Image, display

from typing import TypedDict, Optional, List, Dict, Any, Annotated, Tuple, Optional, Literal, Callable
from typing_extensions import TypedDict
import operator

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState
from langgraph.graph import StateGraph, START, END

from langgraph.prebuilt import tools_condition, ToolNode
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, BaseMessage
from langchain_openai import ChatOpenAI

from langgraph.graph.message import add_messages

### NEW
from datetime import datetime, timedelta, date
import pandas as pd
from langchain.tools import tool
from langchain_core.tools import StructuredTool
from langchain.agents import create_agent
from datetime import date, datetime, timedelta
import pandas as pd
from __future__ import annotations
from langchain_core.tools import tool
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import csv
import re
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
import json
from langchain_core.documents import Document

load_dotenv(find_dotenv())

LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")
LANGCHAIN_PROJECT = os.getenv("LANGCHAIN_PROJECT")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LANGCHAIN_TRACING_V2 = os.getenv("LANGCHAIN_TRACING_V2") == "true"

KB_DIR = Path.cwd() / "kb"
BusinessMarketing_KB_PATH = KB_DIR / "BusinessMarketing"

CARDIOLOGY_SCHEDULE_CSV = KB_DIR / "cardiology_appointment_slots.csv"

CARDIOLOGY_KB_PATH = "./kb/cardiology_kb.jsonl"

REBUILD_CHROMA = False   # <-- set to False to reuse persisted DB
CHROMA_DIR = "./chroma_kb"
#CHROMA_DIR = "./chroma_kb_rebuild" if REBUILD_CHROMA else "./chroma_kb"

print("LANGCHAIN_PROJECT:", LANGCHAIN_PROJECT)
print("LANGCHAIN_TRACING_V2:", LANGCHAIN_TRACING_V2)

In [ ]:
from __future__ import annotations

from typing import Any, Dict, List, Optional, TypedDict
from datetime import date, timedelta, datetime
import re
import sqlite3

from pydantic import BaseModel, Field

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver


DB_PATH = BusinessMarketing_KB_PATH / "brand_feedback.db"


# ----------------------------
# Schemas
# ----------------------------

class FeedbackFilters(BaseModel):
    campaign_ids: Optional[List[str]] = None
    feedback_channels: Optional[List[str]] = None
    start_date: Optional[str] = None   # ISO YYYY-MM-DD
    end_date: Optional[str] = None     # ISO YYYY-MM-DD
    timeframe_label: str = "last_4_weeks"


class ThemeStats(BaseModel):
    theme: str
    n: int
    share: float
    avg_comment_len: float
    pos_share: float
    neu_share: float
    neg_share: float
    salience: float


class Adjustment(BaseModel):
    title: str
    change: str
    why_grounded: str
    receipts: List[str]


class AdjustmentsLLMOutput(BaseModel):
    adjustments: List[Adjustment] = Field(..., min_length=3, max_length=3)


# ----------------------------
# Deterministic helpers
# ----------------------------

def _safe_div(a: float, b: float) -> float:
    return a / b if b else 0.0

def _compute_salience(share: float, avg_len: float, overall_avg_len: float, neg_share: float) -> float:
    length_factor = _safe_div(avg_len, overall_avg_len) if overall_avg_len else 1.0
    return share * length_factor * (1.0 + 0.5 * neg_share)

def _select_focus_themes(theme_stats: List[ThemeStats], k: int = 4) -> List[str]:
    if not theme_stats:
        return []
    by_sal = sorted(theme_stats, key=lambda t: t.salience, reverse=True)
    focus = [t.theme for t in by_sal[:k]]

    neg_sorted = sorted(theme_stats, key=lambda t: t.neg_share, reverse=True)
    if neg_sorted and neg_sorted[0].neg_share >= 0.5 and neg_sorted[0].theme not in focus:
        focus[-1] = neg_sorted[0].theme

    mixed = [t for t in theme_stats if t.pos_share > 0.2 and t.neg_share > 0.2]
    mixed_sorted = sorted(mixed, key=lambda t: t.salience, reverse=True)
    if mixed_sorted and mixed_sorted[0].theme not in focus:
        focus[-1] = mixed_sorted[0].theme

    out = []
    for x in focus:
        if x not in out:
            out.append(x)
    return out


# ----------------------------
# Tools: SQLite retrieval + deterministic aggregation
# ----------------------------

@tool
def read_campaign_feedback(
    campaign_ids: Optional[List[str]] = None,
    feedback_channels: Optional[List[str]] = None,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
) -> List[Dict[str, Any]]:
    """
    SQLite-backed feedback retrieval.
    """

    where = []
    params: List[Any] = []

    if campaign_ids:
        where.append(f"campaign_id IN ({','.join(['?'] * len(campaign_ids))})")
        params.extend(campaign_ids)

    if feedback_channels:
        where.append(f"feedback_channel IN ({','.join(['?'] * len(feedback_channels))})")
        params.extend([c.lower() for c in feedback_channels])

    if start_date:
        where.append("created_at >= ?")
        params.append(start_date)

    if end_date:
        where.append("created_at <= ?")
        params.append(end_date)

    where_sql = "WHERE " + " AND ".join(where) if where else ""

    sql = f"""
        SELECT feedback_id, campaign_id, feedback_channel, sentiment,
               primary_theme, comment_length_words, created_at
        FROM campaign_feedback
        {where_sql}
    """

    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()
    cur.execute(sql, params)

    rows = [dict(r) for r in cur.fetchall()]
    conn.close()

    return rows


def aggregate_themes(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Deterministic aggregation: counts + sentiment mix + avg length + salience.
    """
    total_n = len(rows)
    if total_n == 0:
        return {"total_n": 0, "overall_avg_len": 0.0, "themes": []}

    overall_avg_len = sum(int(r.get("comment_length_words") or 0) for r in rows) / total_n

    by_theme: Dict[str, List[Dict[str, Any]]] = {}
    for r in rows:
        theme = (r.get("primary_theme") or "").strip() or "UNKNOWN_THEME"
        by_theme.setdefault(theme, []).append(r)

    themes_out = []
    for theme, grp in by_theme.items():
        n = len(grp)
        share = n / total_n
        avg_len = sum(int(g.get("comment_length_words") or 0) for g in grp) / n

        pos = sum(1 for g in grp if (g.get("sentiment") or "").lower() == "positive")
        neu = sum(1 for g in grp if (g.get("sentiment") or "").lower() == "neutral")
        neg = sum(1 for g in grp if (g.get("sentiment") or "").lower() == "negative")

        pos_share = pos / n
        neu_share = neu / n
        neg_share = neg / n

        salience = _compute_salience(share, avg_len, overall_avg_len, neg_share)

        themes_out.append({
            "theme": theme,
            "n": n,
            "share": round(share, 4),
            "avg_comment_len": round(avg_len, 2),
            "pos_share": round(pos_share, 4),
            "neu_share": round(neu_share, 4),
            "neg_share": round(neg_share, 4),
            "salience": round(salience, 6),
        })

    themes_out.sort(key=lambda x: x["salience"], reverse=True)
    return {"total_n": total_n, "overall_avg_len": round(overall_avg_len, 2), "themes": themes_out}


# ----------------------------
# LangGraph State + Nodes
# ----------------------------

class FeedbackAgentState(TypedDict, total=False):
    messages: List[BaseMessage]
    filters: Dict[str, Any]
    feedback_rows: List[Dict[str, Any]]
    aggregation: Dict[str, Any]
    focus_themes: List[str]
    adjustments: List[Dict[str, Any]]


def _latest_user_text(state: FeedbackAgentState) -> str:
    for m in reversed(state.get("messages", [])):
        if isinstance(m, HumanMessage):
            return m.content
    return ""


def _infer_date_range(text: str) -> Dict[str, Optional[str]]:
    """
    Simple deterministic timeframe parsing:
      - "last 4 weeks" / "last N weeks"
      - "last month" -> 4 weeks
      - default -> last 4 weeks
    Uses today's date.
    """
    today = date.today()
    tl = text.lower()

    weeks = 4
    m = re.search(r"last\s+(\d+)\s+weeks?", tl)
    if m:
        weeks = max(1, int(m.group(1)))
    elif "last month" in tl:
        weeks = 4
    elif "last 8 weeks" in tl:
        weeks = 8

    start = today - timedelta(days=7 * weeks)
    return {
        "start_date": start.isoformat(),
        "end_date": today.isoformat(),
        "timeframe_label": f"last_{weeks}_weeks"
    }


def parse_request_node(state: FeedbackAgentState) -> FeedbackAgentState:
    text = _latest_user_text(state)

    campaign_ids = re.findall(r"\bCAMP[_-]?\d+\b", text.upper())
    channels = [ch for ch in ["email", "app", "social", "web"] if re.search(rf"\b{ch}\b", text.lower())]

    dr = _infer_date_range(text)

    filters = FeedbackFilters(
        campaign_ids=campaign_ids or None,
        feedback_channels=channels or None,
        start_date=dr["start_date"],
        end_date=dr["end_date"],
        timeframe_label=dr["timeframe_label"],
    ).model_dump()

    return {"filters": filters}


def retrieve_node(state: FeedbackAgentState) -> FeedbackAgentState:
    rows = read_campaign_feedback.invoke(state["filters"])
    return {"feedback_rows": rows}


def aggregate_node(state: FeedbackAgentState) -> FeedbackAgentState:
    agg = aggregate_themes(state["feedback_rows"])
    return {"aggregation": agg}


def select_focus_node(state: FeedbackAgentState) -> FeedbackAgentState:
    stats = [ThemeStats(**t) for t in state["aggregation"].get("themes", [])]
    focus = _select_focus_themes(stats, k=4)
    return {"focus_themes": focus}


def recommend_node(state: FeedbackAgentState) -> FeedbackAgentState:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    themes = state["aggregation"].get("themes", [])
    focus = set(state.get("focus_themes", []))
    focus_rows = [t for t in themes if t["theme"] in focus] or themes[:4]

    system = (
        "You are a brand/content strategy assistant.\n"
        "Return EXACTLY 3 content adjustments.\n"
        "Content-only changes: copy, creative, messaging structure, framing, CTA, sequencing.\n"
        "Ground strictly in provided theme stats. Do NOT introduce new themes. Do NOT claim causes.\n"
        "Each adjustment must include 2–3 receipts citing actual numbers (n/share/neg_share/pos_share/avg_len).\n"
        "Output must be JSON: {\"adjustments\": [...]} with fields title, change, why_grounded, receipts.\n"
    )
    user = f"Theme stats (sorted by salience):\n{themes}\n\nFocus themes:\n{focus_rows}\n"

    structured = llm.with_structured_output(AdjustmentsLLMOutput)
    out = structured.invoke([("system", system), ("user", user)])
    return {"adjustments": [a.model_dump() for a in out.adjustments]}


def format_node(state: FeedbackAgentState) -> FeedbackAgentState:
    f = FeedbackFilters(**state["filters"])
    rows_n = len(state.get("feedback_rows", []))
    themes = state["aggregation"].get("themes", [])
    focus = state.get("focus_themes", [])
    adjustments = state.get("adjustments", [])

    lines = []
    lines.append("Brand Manager Feedback Themes Summary\n")
    lines.append(f"Applied filters: campaign_ids={f.campaign_ids or 'ALL'}, channels={f.feedback_channels or 'ALL'}, "
                 f"date_range={f.start_date}..{f.end_date} ({f.timeframe_label})")
    lines.append(f"Total feedback rows analyzed: {rows_n}\n")

    if rows_n == 0:
        lines.append("No feedback matched these filters. Try widening the date range or removing channel/campaign filters.")
        return {"messages": [AIMessage(content="\n".join(lines))]}

    lines.append("Top themes (by salience):")
    for t in themes[:6]:
        lines.append(
            f"- {t['theme']}: n={t['n']}, share={t['share']:.2f}, neg={t['neg_share']:.2f}, pos={t['pos_share']:.2f}, "
            f"avg_len={t['avg_comment_len']:.1f}, salience={t['salience']:.4f}"
        )

    if focus:
        lines.append(f"\nFocus themes used for recommendations: {', '.join(focus)}")

    lines.append("\n3 content adjustments (grounded):")
    for i, a in enumerate(adjustments, 1):
        lines.append(f"\n{i}) {a['title']}")
        lines.append(f"   Change: {a['change']}")
        lines.append(f"   Why (grounded): {a['why_grounded']}")
        lines.append("   Receipts:")
        for r in a["receipts"]:
            lines.append(f"   - {r}")

    lines.append("\nLimitations:")
    lines.append("- Recommendations are content hypotheses grounded only in aggregated theme + sentiment patterns, not causal explanations.")
    lines.append("- No new themes were inferred beyond primary_theme; if taxonomy is coarse, insights will be correspondingly coarse.")

    return {"messages": [AIMessage(content="\n".join(lines))]}


def build_graph():
    g = StateGraph(FeedbackAgentState)
    g.add_node("parse", parse_request_node)
    g.add_node("retrieve", retrieve_node)
    g.add_node("aggregate", aggregate_node)
    g.add_node("focus", select_focus_node)
    g.add_node("recommend", recommend_node)
    g.add_node("format", format_node)

    g.set_entry_point("parse")
    g.add_edge("parse", "retrieve")
    g.add_edge("retrieve", "aggregate")
    g.add_edge("aggregate", "focus")
    g.add_edge("focus", "recommend")
    g.add_edge("recommend", "format")
    g.add_edge("format", END)

    return g.compile(checkpointer=MemorySaver())

graph = build_graph()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

<br><br><br>

<hr style="border:30px solid coral "> </hr>
<hr style="border:2px solid coral "> </hr>


# Demonstration

<hr style="border:2px solid coral "> </hr>


In [ ]:
out = graph.invoke({"messages": [HumanMessage(content="Summarize last month feedback for CAMP_105 across app and email; suggest 3 content adjustments.")]},
                   config={"configurable": {"thread_id": "demo"}})
print(out["messages"][-1].content)